# Tutorial Prático: Funcionalidades Principais do Módulo `fermion.py` no Ket

Este notebook demonstra as principais funções do módulo fermiônico do Ket (`ket.fermion`), cobrindo a criação de operadores, ordenação normal (normal ordering), simplificação numérica, adjunto hermitiano e verificadores de simetrias físicas (N e Sz).

## 1. Operadores de Criação e Aniquilação (`CreateFermion` e `AnnihilateFermion`)

Os operadores fundamentais a_i⁺ (criação) e a_i (aniquilação) podem ser instanciados informando o índice do orbital. O Ket atribui automaticamente o spin alpha (spin-up, em orbitais pares) e spin beta (spin-down, em orbitais ímpares), ou aceita a atribuição explícita `spin="a"` ou `spin="b"`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../src"))
import ket
from ket import CreateFermion, AnnihilateFermion, Fermion, FermionSentence

# Operador de criação no orbital 0 (spin alpha por padrão)
a0_dag = CreateFermion(0)
print("Criação orbital 0:", a0_dag)

# Operador de aniquilação no orbital 1 (spin beta por padrão)
a1_ann = AnnihilateFermion(1)
print("Aniquilação orbital 1:", a1_ann)

# Atribuição explícita de spin
a0_beta_dag = CreateFermion(0, spin="b")
print("Criação orbital 0 com spin beta explicitado:", a0_beta_dag)

## 2. Produtos e Combinações Lineares (`Fermion` e `FermionSentence`)

Multiplicar operadores gera um termo do tipo `Fermion` (produto de operadores). Somar e multiplicar por escalares gera uma `FermionSentence` (combinação linear ponderada).

In [ ]:
# Termo de acoplamento (hopping): a⁺_0 a_1
hopping_term = CreateFermion(0) * AnnihilateFermion(1)
print("Termo de hopping:", hopping_term)

# Combinação linear (FermionSentence) com coeficientes reais e complexos
hamiltonian_simples = 1.5 * (CreateFermion(0) * AnnihilateFermion(0)) + (0.2 + 0.5j) * hopping_term
print("
Expressão fermiônica (FermionSentence):")
print(hamiltonian_simples)

## 3. Ordenação Normal (`normal_ordered`)

A ordenação normal reorganiza os operadores de modo que todos os operadores de criação (a⁺) precedam os de aniquilação (a), ordenados do maior índice para o menor, aplicando as relações de anticomutação dos férmions.
Por exemplo, a_0 a⁺_0 é transformado em I - a⁺_0 a_0.

In [ ]:
# Termo NÃO ordenado: a_0 a⁺_0
nao_ordenado = FermionSentence({AnnihilateFermion(0) * CreateFermion(0): 1.0})
print("Expressão NÃO ordenada:")
print(nao_ordenado)

# Aplicando a ordenação normal
ordenado = nao_ordenado.normal_ordered()
print("
Após normal_ordered() (Resultado esperado: I - a⁺_0 a_0):")
print(ordenado)

## 4. Simplificação por Tolerância Numérica (`simplify`)

O método `.simplify(tol)` elimina termos com coeficientes cujos módulos são inferiores à tolerância `tol` (padrão 10⁻¹⁰), limpando ruídos numéricos de simulação.

In [ ]:
# Criando uma sentença com um termo relevante e um ruído numérico insignificante
ruido = 1e-12 * CreateFermion(2)
termo_valido = 2.0 * CreateFermion(0)

sentence = termo_valido + ruido
print("Antes da simplificação:")
print(sentence)

# Aplicando a simplificação
sentence.simplify()
print("
Após sentence.simplify():")
print(sentence)

## 5. Operador Adjunto Hermitiano (`adjoint`)

O método `.adjoint()` calcula o conjugado hermitiano (A)⁺ de uma expressão fermiônica: ele inverte a ordem dos operadores e toma o conjugado complexo de cada coeficiente.

In [ ]:
# Definindo uma expressão com coeficiente complexo: (1 + 2j) * a⁺_0 a_1
expr = (1 + 2j) * (CreateFermion(0) * AnnihilateFermion(1))
print("Expressão original (A):", expr)

# Adjunto Hermitiano (A⁺)
expr_adj = expr.adjoint()
print("Adjunto Hermitiano (A⁺):", expr_adj)

## 6. Operador Número de Partículas (`number_operator`)

A função `ket.number_operator(n_orbitals)` gera automaticamente o operador número total N = ∑ a_i⁺ a_i para um número de orbitais, ou para um orbital específico.

In [ ]:
# Operador número total para 4 orbitais
n_total = ket.number_operator(4)
print("Operador Número Total (4 orbitais):")
print(n_total)

# Operador número apenas no orbital 2
n_orb2 = ket.number_operator(4, orbital=2)
print("
Operador Número no orbital 2:")
print(n_orb2)

## 7. Verificadores de Simetrias Físicas (`conserves_particle_number` e `conserves_spin_z`)

O Ket possui métodos automáticos para inspecionar se uma expressão fermiônica preserva o número de elétrons (N) e a projeção de spin (Sz).

In [ ]:
# Termo que conserva partículas: a⁺_0 a_1
termo_conserva_n = FermionSentence({CreateFermion(0) * AnnihilateFermion(1): 1.0})
# Termo que NÃO conserva partículas: apenas criação a⁺_0
termo_cria_particula = FermionSentence({CreateFermion(0): 1.0})

print("termo_conserva_n conserva número de partículas?", termo_conserva_n.conserves_particle_number())
print("termo_cria_particula conserva número de partículas?", termo_cria_particula.conserves_particle_number())

# Conservação de Spin Sz:
# a⁺_0 (alpha) a_2 (alpha) -> conserva Sz (mesmo spin)
termo_mesmo_spin = FermionSentence({CreateFermion(0) * AnnihilateFermion(2): 1.0})
# a⁺_0 (alpha) a_1 (beta) -> troca de spin (spin-flip, altera Sz)
termo_spin_flip = FermionSentence({CreateFermion(0) * AnnihilateFermion(1): 1.0})

print("
termo_mesmo_spin conserva Sz?", termo_mesmo_spin.conserves_spin_z())
print("termo_spin_flip conserva Sz?", termo_spin_flip.conserves_spin_z())